## Data Cleaning Principles

- Raw data is not modified; all transformations are applied on a copy.
- Returns and cancellations are **not deleted**, but flagged.
- Missing data is handled deliberately, not blindly removed.
- All assumptions are documented and justified.
- Business meaning is prioritized over aggressive data reduction.

### PHASE 1: Data Preparation

- Load year-wise CSV files

- Combine into a single dataset

- Create a working copy

- Record baseline row count

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df_2009 = pd.read_csv("/Users/hepigediya/Desktop/retail-omnichannel-analytics/data/raw/online_retail_2009_2010.csv")
df_2010 = pd.read_csv("/Users/hepigediya/Desktop/retail-omnichannel-analytics/data/raw/online_retail_2010_2011.csv")

In [3]:
df_raw = pd.concat([df_2009, df_2010], ignore_index=True)

In [4]:
df = df_raw.copy()

In [5]:
df.shape

(1067371, 8)

### PHASE 2: Schema Standardization

- Standardize column names (snake_case)

- Ensure correct data types

- Convert dates to datetime format

In [6]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

Standardize Column Names
 
 Rules we apply
- Lowercase
- Replace spaces with _
- Remove special characters
- Use snake_case

In [7]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("-", "_")
)

Rename Key Columns (Explicit Mapping)

Some columns should be business-clear.

In [8]:
df = df.rename(columns={
    "invoice": "invoice_no",
    "stockcode": "stock_code",
    "invoicedate": "invoice_date",
    "price": "unit_price",
    "customerid": "customer_id"
})

In [9]:
df.columns

Index(['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date',
       'unit_price', 'customer_id', 'country'],
      dtype='object')

In [10]:
df.dtypes

invoice_no       object
stock_code       object
description      object
quantity          int64
invoice_date     object
unit_price      float64
customer_id     float64
country          object
dtype: object

In [11]:
df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')

In [12]:
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')

In [13]:
df['customer_id'] = pd.to_numeric(df['customer_id'], errors='coerce')# Enforce nullable integer type for customer_id
df['customer_id'] = df['customer_id'].astype('Int64')

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice_no    1067371 non-null  object        
 1   stock_code    1067371 non-null  object        
 2   description   1062989 non-null  object        
 3   quantity      1067371 non-null  int64         
 4   invoice_date  1067371 non-null  datetime64[ns]
 5   unit_price    1067371 non-null  float64       
 6   customer_id   824364 non-null   Int64         
 7   country       1067371 non-null  object        
dtypes: Int64(1), datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 66.2+ MB


In [15]:
# Missing Value Assessment

print("\nMissing Values After Type Conversion:")
missing_df = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().sum() / len(df) * 100).round(2)
})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)
if len(missing_df) > 0:
    print(missing_df.to_string(index=False))
else:
    print("✓ No missing values detected")


Missing Values After Type Conversion:
     column  missing_count  missing_pct
customer_id         243007        22.77
description           4382         0.41


### PHASE 3: Returns & Cancellations Handling

- Identify negative quantity values

- Identify invoices starting with 'C'

- Create flags:

- is_return

- is_cancellation

- Do not remove these records

In [16]:
# Flag returned items based on negative quantity
df['is_return'] = df['quantity'] < 0

In [17]:
# Flag cancellation / credit invoices (invoice numbers starting with 'C')
df['is_cancellation'] = df['invoice_no'].astype(str).str.startswith('C')

In [18]:
# Count return and cancellation flags
df['is_return'].value_counts()

is_return
False    1044421
True       22950
Name: count, dtype: int64

In [19]:
df['is_cancellation'].value_counts()

is_cancellation
False    1047877
True       19494
Name: count, dtype: int64

In [20]:
# Count return and cancellation flags
df[['is_return', 'is_cancellation']].value_counts()

is_return  is_cancellation
False      False              1044420
True       True                 19493
           False                 3457
False      True                     1
Name: count, dtype: int64

In [21]:
# Understand relationship between returns and cancellations
pd.crosstab(df['is_return'], df['is_cancellation'])

is_cancellation,False,True
is_return,,
False,1044420,1
True,3457,19493


In [22]:
# Return-Cancellation Deep Dive

print("\nReturn vs Cancellation Cross-tabulation:")
print(pd.crosstab(df['is_return'], df['is_cancellation'], margins=True))

print("\nRecords with BOTH return AND cancellation flags:")
both_flags = (df['is_return'] & df['is_cancellation']).sum()
print(f"  Count: {both_flags:,}")

# Sample records that are both return and cancellation
if both_flags > 0:
    print("\nSample records with both flags:")
    print(df[df['is_return'] & df['is_cancellation']][['invoice_no', 'quantity', 'unit_price']].head())


Return vs Cancellation Cross-tabulation:
is_cancellation    False   True      All
is_return                               
False            1044420      1  1044421
True                3457  19493    22950
All              1047877  19494  1067371

Records with BOTH return AND cancellation flags:
  Count: 19,493

Sample records with both flags:
    invoice_no  quantity  unit_price
178    C489449       -12        2.95
179    C489449        -6        1.65
180    C489449        -4        4.25
181    C489449        -6        2.10
182    C489449       -12        2.95


In [23]:
# Number of return rows
(df['quantity'] < 0).sum()

22950

In [24]:
# Number of cancellation invoices
df['invoice_no'].astype(str).str.startswith('C').sum()

19494

### PHASE 4: Customer Data Handling

- Identify missing customer_id

- Retain anonymous transactions

- Document impact on customer-level analysis

In [25]:
# Count missing customer IDs
df['customer_id'].isna().sum()

243007

In [26]:
# Flag anonymous customer transactions
df['is_anonymous_customer'] = df['customer_id'].isna()

In [27]:
# Distribution of anonymous vs known customers
df['is_anonymous_customer'].value_counts()

is_anonymous_customer
False    824364
True     243007
Name: count, dtype: int64

In [28]:
# Percentage of anonymous transactions
(df['is_anonymous_customer'].mean() * 100).round(2)

22.77

### PHASE 5: Pricing Validation

- Identify zero or negative unit_price

- Remove invalid pricing rows

- Justify removal as data quality issue

In [29]:
# Identify rows with zero or negative unit price
invalid_price_mask = df['unit_price'] <= 0

invalid_price_mask.sum()

6207

In [30]:
# Preview invalid pricing records
df.loc[invalid_price_mask].head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,is_return,is_cancellation,is_anonymous_customer
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,True,False,True
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,<NA>,United Kingdom,True,False,True
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,<NA>,United Kingdom,True,False,True
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,<NA>,United Kingdom,True,False,True
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,<NA>,United Kingdom,True,False,True


In [31]:
# Remove invalid pricing rows
df = df.loc[~invalid_price_mask].copy()

In [32]:
df.shape

(1061164, 11)

In [33]:
# Confirm all remaining prices are positive
(df['unit_price'] <= 0).sum()

0

### PHASE 6: Feature Engineering

- Create derived fields:

- sales_amount = quantity × unit_price

- order_date

- order_month

- order_year

In [34]:
# Create sales amount (revenue) at line-item level
df['sales_amount'] = df['quantity'] * df['unit_price']

In [35]:
# Extract order date
df['order_date'] = df['invoice_date'].dt.date

In [36]:
# Extract order month
df['order_month'] = df['invoice_date'].dt.month

In [37]:
# Extract order year
df['order_year'] = df['invoice_date'].dt.year

In [38]:
df[['sales_amount', 'order_date', 'order_month', 'order_year']].head()

,sales_amount,order_date,order_month,order_year
0,83.4,2009-12-01,12,2009
1,81.0,2009-12-01,12,2009
2,81.0,2009-12-01,12,2009
3,100.8,2009-12-01,12,2009
4,30.0,2009-12-01,12,2009


In [39]:
# Outlier Detection Summary

print("\nOutlier Summary (Informational):")
print(f"  Max quantity (single transaction):  {df['quantity'].max():>10.0f}")
print(f"  Min quantity (single transaction):  {df['quantity'].min():>10.0f}")
print(f"  Max unit price:                     £{df['unit_price'].max():>13.2f}")
print(f"  Min unit price:                     £{df['unit_price'].min():>13.2f}")
print(f"  Max sales amount:                   £{df['sales_amount'].max():>13.2f}")
print(f"  Min sales amount:                   £{df['sales_amount'].min():>13.2f}")

# Identify bulk transactions (top 1% by quantity)
bulk_threshold = df['quantity'].quantile(0.99)
df['is_bulk_transaction'] = df['quantity'] > bulk_threshold
print(f"\n  Bulk transactions (qty > {bulk_threshold:.0f}): {df['is_bulk_transaction'].sum():>10,}")


Outlier Summary (Informational):
  Max quantity (single transaction):       80995
  Min quantity (single transaction):      -80995
  Max unit price:                     £     38970.00
  Min unit price:                     £         0.00
  Max sales amount:                   £    168469.60
  Min sales amount:                   £   -168469.60

  Bulk transactions (qty > 100):     10,294


### PHASE 7: Duplicate Validation

- Check for exact duplicate rows

- Remove duplicates only if confirmed

In [40]:
# Check number of exact duplicate rows
df.duplicated().sum()

34147

In [41]:
# View sample duplicate rows
df[df.duplicated()].head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,is_return,is_cancellation,is_anonymous_customer,sales_amount,order_date,order_month,order_year,is_bulk_transaction
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom,False,False,False,3.75,2009-12-01,12,2009,False
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329,United Kingdom,False,False,False,5.10,2009-12-01,12,2009,False
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329,United Kingdom,False,False,False,7.80,2009-12-01,12,2009,False
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom,False,False,False,3.75,2009-12-01,12,2009,False
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom,False,False,False,3.75,2009-12-01,12,2009,False


In [42]:
# Remove exact duplicate rows
df = df.drop_duplicates().copy()

In [43]:
# Confirm no duplicates remain
df.duplicated().sum()

0

In [44]:
df.shape

(1027017, 16)

### PHASE 8: Channel Preparation (Later Use)

- Keep structure ready for sales_channel

- Channel logic will be derived in analysis phase

In [45]:
# Create placeholder column for future channel derivation
df['sales_channel'] = None

In [46]:
df[['sales_channel']].head()

,sales_channel
0,None
1,None
2,None
3,None
4,None


### PHASE 9: Validation & Export

- Validate row counts after each step

- Sanity-check key metrics

- Export cleaned dataset to:

data/processed/cleaned_retail_sales.csv

In [47]:
# Final row count after all cleaning steps
final_row_count = df.shape[0]
final_row_count

1027017

In [48]:
df = df.reset_index(drop=True)

In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1027017 entries, 0 to 1027016
Data columns (total 17 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   invoice_no             1027017 non-null  object        
 1   stock_code             1027017 non-null  object        
 2   description            1027017 non-null  object        
 3   quantity               1027017 non-null  int64         
 4   invoice_date           1027017 non-null  datetime64[ns]
 5   unit_price             1027017 non-null  float64       
 6   customer_id            797815 non-null   Int64         
 7   country                1027017 non-null  object        
 8   is_return              1027017 non-null  bool          
 9   is_cancellation        1027017 non-null  bool          
 10  is_anonymous_customer  1027017 non-null  bool          
 11  sales_amount           1027017 non-null  float64       
 12  order_date             10270

In [50]:
# Save cleaning metadata for audit trail
import json
from datetime import datetime

metadata = {
    'cleaned_at': datetime.now().isoformat(),
    'raw_row_count': df_raw.shape[0],
    'final_row_count': df.shape[0],
    'rows_removed': df_raw.shape[0] - df.shape[0],
    'removal_percentage': round((df_raw.shape[0] - df.shape[0]) / df_raw.shape[0] * 100, 2),
    'date_range': {
        'start': str(df['invoice_date'].min().date()),
        'end': str(df['invoice_date'].max().date())
    },
    'total_revenue': float(df['sales_amount'].sum()),
    'unique_customers': int(df['customer_id'].nunique()),
    'unique_products': int(df['stock_code'].nunique()),
    'flags': {
        'returns': int(df['is_return'].sum()),
        'cancellations': int(df['is_cancellation'].sum()),
        'anonymous': int(df['is_anonymous_customer'].sum())
    }
}

metadata_path = '/Users/hepigediya/Desktop/retail-omnichannel-analytics/data/processed/cleaning_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✓ Metadata saved: {metadata_path}")


✓ Metadata saved: /Users/hepigediya/Desktop/retail-omnichannel-analytics/data/processed/cleaning_metadata.json


In [51]:
### PHASE 9: Validation & Export Summary

print("\n" + "="*70)
print("DATA CLEANING IMPACT SUMMARY")
print("="*70)
print(f"Raw records loaded:              {df_raw.shape[0]:>10,}")
print(f"Records after invalid pricing:   {df.shape[0]:>10,}")
print(f"Records removed:                 {df_raw.shape[0] - df.shape[0]:>10,}")
print(f"Removal rate:                    {((df_raw.shape[0] - df.shape[0]) / df_raw.shape[0] * 100):>9.2f}%")
print()
print(f"Final dataset dimensions:        {df.shape[0]:>10,} rows × {df.shape[1]:>2} columns")
print()
print("Flag Distribution:")
print(f"  Returns (negative qty):        {df['is_return'].sum():>10,} ({df['is_return'].mean()*100:.2f}%)")
print(f"  Cancellations (invoice C):     {df['is_cancellation'].sum():>10,} ({df['is_cancellation'].mean()*100:.2f}%)")
print(f"  Anonymous customers:           {df['is_anonymous_customer'].sum():>10,} ({df['is_anonymous_customer'].mean()*100:.2f}%)")
print()
print("Key Business Metrics:")
print(f"  Date range:                    {df['invoice_date'].min().date()} to {df['invoice_date'].max().date()}")
print(f"  Total revenue (all):           £{df['sales_amount'].sum():>14,.2f}")
print(f"  Total revenue (sales only):    £{df.loc[~df['is_return'] & ~df['is_cancellation'], 'sales_amount'].sum():>14,.2f}")
print(f"  Unique customers:              {df['customer_id'].nunique():>10,}")
print(f"  Unique products:               {df['stock_code'].nunique():>10,}")
print(f"  Unique invoices:               {df['invoice_no'].nunique():>10,}")
print("="*70)


DATA CLEANING IMPACT SUMMARY
Raw records loaded:               1,067,371
Records after invalid pricing:    1,027,017
Records removed:                     40,354
Removal rate:                         3.78%

Final dataset dimensions:         1,027,017 rows × 17 columns

Flag Distribution:
  Returns (negative qty):            19,103 (1.86%)
  Cancellations (invoice C):         19,104 (1.86%)
  Anonymous customers:              229,202 (22.32%)

Key Business Metrics:
  Date range:                    2009-12-01 to 2011-12-09
  Total revenue (all):           £ 19,014,209.84
  Total revenue (sales only):    £ 20,476,260.45
  Unique customers:                   5,939
  Unique products:                    4,932
  Unique invoices:                   48,369


In [52]:
# Export cleaned dataset
output_path = '/Users/hepigediya/Desktop/retail-omnichannel-analytics/data/processed/cleaned_retail_sales.csv'
df.to_csv(output_path, index=False)

print(f"\n" + "="*70)
print("✓ EXPORT COMPLETE")
print("="*70)
print(f"File saved: {output_path}")
print(f"Dimensions: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("="*70)


✓ EXPORT COMPLETE
File saved: /Users/hepigediya/Desktop/retail-omnichannel-analytics/data/processed/cleaned_retail_sales.csv
Dimensions: 1,027,017 rows × 17 columns
